# 06 — QLoRA Fine-Tune on 7B Decoder (Stretch — Colab GPU)

**This notebook requires a free Google Colab T4 or L4 GPU.**
Run on Colab: open this file in Colab or copy cells.

Goal: Fine-tune a 7B-parameter decoder model (Llama-3 8B or Mistral 7B) on work order classification
using QLoRA (4-bit NF4 quantization + LoRA adapters via bitsandbytes).

This demonstrates:
- Running a 7B parameter model on free Colab GPU (fits in 16 GB VRAM with 4-bit quant)
- QLoRA: LoRA on quantized model — combines memory savings + parameter efficiency
- Adapter weights are still small even though base model is 7B parameters

**Note:** `bitsandbytes` requires Linux/CUDA — not available on Mac. This must run on Colab.

In [ ]:
# Run this first cell on Colab to install required packages
# !pip install transformers peft bitsandbytes accelerate datasets -q

# Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. This notebook requires a CUDA GPU.')

In [ ]:
# Upload or generate data on Colab
# Option A: upload work_orders.csv to Colab via Files panel
# Option B: run the generator inline
import pandas as pd
import sys
# sys.path.insert(0, '/content/maintenance_nlp')  # adjust to your Colab path

# from data.synthetic_generator import generate_corpus
# df, _ = generate_corpus()
# print(df.head(2))

In [ ]:
# Load quantized Mistral 7B
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType
import torch

MODEL_ID = 'mistralai/Mistral-7B-v0.1'  # or 'meta-llama/Meta-Llama-3-8B'
CATEGORY_LABELS = [
    'mechanical_failure', 'electrical_failure', 'hydraulic_failure',
    'instrumentation_failure', 'preventive_maintenance', 'operator_damage',
]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

# NOTE: This requires HuggingFace login for gated models (Llama-3)
# !huggingface-cli login

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    num_labels=len(CATEGORY_LABELS),
    device_map='auto',
)

In [ ]:
# Apply LoRA on top of quantized model (QLoRA)
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj'],
    bias='none',
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()
# Expected: ~0.1–0.2% of 7B parameters — about 4–8M trainable params

In [ ]:
# Fine-tuning setup (same Trainer pattern as notebook 05)
# ... (standard Trainer code — omitted for brevity; follow notebook 05 pattern)
# After training:
# model.save_pretrained('maintenance_nlp_qlora_adapter')

print('QLoRA notebook scaffolded. Run on Colab for actual training.')
print('Key hyperparameters to record:')
print('  - Base model size, quantized memory footprint')
print('  - Trainable params (should be <1%)')
print('  - F1 vs. DistilBERT LoRA')
print('  - GPU-minutes on free T4')
print('  - Adapter size (should be ~few MB even for 7B base model)')